# Ex.5: A minimal RAG pipeline
**EECE.4860/5860**: ungraded practice for Lecture 9. The smallest useful RAG system: embed documents, retrieve the most similar ones for a question, and let an LLM answer from them. Lab 3 builds on this.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acanets/genai-labs/blob/spring27/exercises/ex5_simple_rag.ipynb)

In [ ]:
import importlib.util
if importlib.util.find_spec("genai_utils") is None:
    get_ipython().run_line_magic("pip", 'install -q "git+https://github.com/acanets/genai-labs@spring27"')
if importlib.util.find_spec("sentence_transformers") is None:
    get_ipython().run_line_magic("pip", "install -q sentence-transformers")
import json, urllib.request, numpy as np, torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from genai_utils import setup_env
device, dtype = setup_env()
URL = "https://raw.githubusercontent.com/acanets/genai-labs/spring27/labs/lab3/data/corpus.jsonl"
docs = [json.loads(l) for l in urllib.request.urlopen(URL).read().decode().splitlines() if "SYNTHETIC" not in l]
print(len(docs), "documents, e.g.:", docs[0]["title"])

## 1. Index: one embedding per document

In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
E = embedder.encode([d["title"] + ". " + d["text"] for d in docs], normalize_embeddings=True)

def retrieve(question, k=3):
    s = E @ embedder.encode([question], normalize_embeddings=True)[0]
    return [docs[i] for i in np.argsort(-s)[:k]]

for d in retrieve("How can a small model make decoding of a large model faster?"):
    print("-", d["id"], d["title"])

## 2. Generate an answer grounded in the retrieved text

In [ ]:
LLM = "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(LLM)
llm = AutoModelForCausalLM.from_pretrained(LLM, dtype=dtype).to(device)

def answer(question):
    ctx = "\n\n".join(f"[{d['id']}] {d['text']}" for d in retrieve(question))
    msgs = [{"role": "user", "content": f"Answer using only these sources and cite them as [id].\n\n{ctx}\n\nQuestion: {question}"}]
    p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    enc = tok(p, return_tensors="pt").to(device)
    out = llm.generate(**enc, max_new_tokens=150, do_sample=False)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

print(answer("How many image-text pairs was CLIP trained on?"))

## 3. Try it
* Ask a question the corpus cannot answer (e.g., today's weather). Does the model admit it?
* Ask the same question without retrieval (`llm` alone). What changes?